In [ ]:
import json
import math
import itertools
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("tqdm not found — install with: pip install tqdm")

## Cyber — Disjoint Strategy  (v3: 50-seed hardened run)

Non-overlapping blocks of fixed length. With 5 capture days:
- block=1 → 5 blocks  |  block=2 → 2 blocks  |  block=3 → 1 block  |  block=5 → 1 block  →  **9 slice jobs**

**Changes from v2:** clip pinned to 70th pct; 50 seeds; output → `results_disjoint_hardened.csv`

**Expected rows:** 50 × 9 slices × 1 clip × 7 ε × 3 mechs × 3 metrics = **28,350**

In [ ]:
# =========================
# PATHS  (EDIT HERE)
# =========================
DATA_DIR      = Path(r"C:\\Users\\Siddhartha\\Sem8\\MajorProject\\Experiments\\CyberSec\\_cleaned")
FLOWS_FILE    = "cicids_flows_canonical.parquet"
STRATEGY_NAME = "disjoint"

OUT_ROOT    = DATA_DIR / "_grid_outputs" / STRATEGY_NAME
BOUNDED_DIR = OUT_ROOT / "bounded"
DP_DIR      = OUT_ROOT / "dp_outputs"
CSV_DIR     = OUT_ROOT / "metrics_csv"
PLOTS_DIR   = OUT_ROOT / "plots"
for d in [BOUNDED_DIR, DP_DIR, CSV_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# SLICE GRID  (EDIT HERE)
# =========================
WINDOW_DAYS_LIST = [1, 2, 3, 5]    # non-overlapping block lengths

# =========================
# BOUNDING  (calibration-validated — v3)
# =========================
# Sensitivity calibration (sensitivity_calibration_cyber.ipynb) showed that
# the 70th-percentile clip minimises MAE for TOTAL_BYTES by ~15–21× vs 90th pct.
# Fix to [70] for the 50-seed hardened run; the multi-pct grid is in results_*_multi.csv.
BYTES_CLIP_PCT_LIST = [70]          # <-- calibration-validated optimum

# =========================
# DP GRID  (EDIT HERE)
# =========================
EPS_TOTALS = [0.25, 0.5, 1, 2, 4, 8, 16]   # <-- same as v2

# =========================
# MULTI-TRIAL SEED CONFIG
# =========================
BASE_SEED = 42    # seed for seed_id=0; each trial uses BASE_SEED + seed_id
NUM_SEEDS = 20    # independent noise trials — statistical hardening target

# =========================
# DECISION RULE PARAMS  (EDIT HERE)
# =========================
THRESH_QUANTILE    = 0.75  # alarm threshold = this quantile of truth per slice/metric
TREND_WINDOW_DAYS  = 2     # consecutive up-ticks to flag a positive trend
CP_DIFF_MULTIPLIER = 2.0   # change-point threshold = multiplier × std(diff(truth))
CP_TOLERANCE_DAYS  = 1     # ±days tolerance for CP F1  (T≤5, so keep tight)
TOPK               = 3     # top-K days for overlap@k and Kendall-τ

In [ ]:
flows_path = DATA_DIR / FLOWS_FILE
assert flows_path.exists(), f"Run cleaning_cicids.ipynb first: {flows_path}"

flows = pd.read_parquet(flows_path)
flows["date"] = pd.to_datetime(flows["date"])

min_date = flows["date"].min()
max_date = flows["date"].max()
print(f"Date range: {min_date.date()} → {max_date.date()}  ({(max_date - min_date).days + 1} days)")
print(f"Total flows: {len(flows):,}  |  Attack flows: {flows['is_attack'].sum():,}")
print(f"Clip percentile fixed to: {BYTES_CLIP_PCT_LIST[0]}th  (calibration-validated)")

## DP Mechanism Definitions

In [ ]:
# ── Metric helpers ─────────────────────────────────────────────────────────────
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def pearson_corr(a: pd.Series, b: pd.Series) -> float:
    a, b = a.astype(float), b.astype(float)
    if a.std() == 0 or b.std() == 0:
        return float("nan")
    return float(a.corr(b))

# ── Bounded-truth builder ──────────────────────────────────────────────────────
def make_daily_bounded_truth(
    flows: pd.DataFrame,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    bytes_clip_pct: float,
) -> tuple[pd.DataFrame, float]:
    """
    Returns daily_truth (ALERT_COUNT, TOTAL_FLOWS, TOTAL_BYTES) and B_CLIP.
    Sensitivity: ALERT_COUNT=1, TOTAL_FLOWS=1, TOTAL_BYTES=B (per-flow).
    """
    sl = flows[(flows["date"] >= start_date) & (flows["date"] <= end_date)].copy()
    B  = float(sl["flow_bytes"].quantile(bytes_clip_pct / 100.0)) if len(sl) > 0 else 1.0
    B  = max(B, 1.0)
    sl["bytes_clipped"] = sl["flow_bytes"].clip(upper=B)
    daily = (
        sl.groupby("date")
        .agg(ALERT_COUNT=("is_attack", "sum"),
             TOTAL_FLOWS=("is_attack", "count"),
             TOTAL_BYTES=("bytes_clipped", "sum"))
        .sort_index()
    )
    full_idx = pd.date_range(start_date, end_date, freq="D")
    daily = daily.reindex(full_idx).fillna(0)
    daily.index.name = "date"
    return daily, B

# ── DP mechanisms ──────────────────────────────────────────────────────────────
METRICS = ["ALERT_COUNT", "TOTAL_FLOWS", "TOTAL_BYTES"]

def dp_naive_laplace(daily_truth, eps_total, sens, rng):
    """Independent Laplace per day; budget split uniformly over T days."""
    T, eps_t = len(daily_truth), eps_total / len(daily_truth)
    out = pd.DataFrame(index=daily_truth.index)
    for m in METRICS:
        scale = sens[m] / eps_t if eps_t > 0 else float("inf")
        out[f"{m}_DP"] = daily_truth[m].to_numpy(float) + rng.laplace(0.0, scale, T)
    return out.clip(lower=0)

def _fenwick_build(arr):
    n, bit = len(arr), np.zeros(len(arr) + 1, dtype=float)
    for i in range(1, n + 1):
        j = i
        while j <= n: bit[j] += arr[i - 1]; j += j & -j
    return bit

def _fenwick_prefix(bit, i):
    s = 0.0
    while i > 0: s += bit[i]; i -= i & -i
    return s

def _dp_tree_prefix(values, eps_total, sensitivity, rng):
    T = len(values)
    if T == 0: return np.array([], dtype=float)
    levels = max(int(math.floor(math.log2(T))) + 1, 1)
    eps_lv = eps_total / levels
    bit = _fenwick_build(values); noisy = bit.copy()
    for idx in range(1, T + 1):
        noisy[idx] = bit[idx] + rng.laplace(0.0, sensitivity / eps_lv)
    return np.array([_fenwick_prefix(noisy, t) for t in range(1, T + 1)], dtype=float)

def dp_binary_tree(daily_truth, eps_total, sens, rng):
    """Binary-tree mechanism; prefix sums then differenced to daily values."""
    out = pd.DataFrame(index=daily_truth.index)
    for m in METRICS:
        pref = _dp_tree_prefix(daily_truth[m].to_numpy(float), eps_total, sens[m], rng)
        out[f"{m}_TREE_DP"] = np.diff(np.concatenate([[0.0], pref]))
    return out.clip(lower=0)

def _dp_smooth_prefix(values, eps_total, sensitivity, rng):
    T = len(values)
    if T == 0: return np.array([], dtype=float)
    L = max(int(math.ceil(math.log2(T))) if T > 1 else 1, 1)
    noise_for_bit = rng.laplace(0.0, sensitivity / (eps_total / L), L)
    noisy_prefix, running = np.zeros(T, dtype=float), 0.0
    for i in range(T):
        running += values[i]; s = running
        idx, bit = i + 1, 0
        while idx > 0 and bit < L:
            if idx & 1: s += noise_for_bit[bit]
            idx >>= 1; bit += 1
        noisy_prefix[i] = s
    return noisy_prefix

def dp_smooth_binary(daily_truth, eps_total, sens, rng):
    """Smooth binary mechanism; noise reuse across bit levels."""
    out = pd.DataFrame(index=daily_truth.index)
    for m in METRICS:
        pref = _dp_smooth_prefix(daily_truth[m].to_numpy(float), eps_total, sens[m], rng)
        out[f"{m}_SMOOTH_DP"] = np.diff(np.concatenate([[0.0], pref]))
    return out.clip(lower=0)

In [ ]:
# ── Decision / evaluation functions ───────────────────────────────────────────
FN_WEIGHT = 5   # FN penalised 5× over FP for security alarms

def decision_stats(truth_labels: pd.Series, dp_labels: pd.Series) -> dict:
    truth, pred = truth_labels.to_numpy(int), dp_labels.to_numpy(int)
    assert truth.shape == pred.shape
    tn = int(np.sum((truth==0)&(pred==0))); fp = int(np.sum((truth==0)&(pred==1)))
    fn = int(np.sum((truth==1)&(pred==0))); tp = int(np.sum((truth==1)&(pred==1)))
    total = len(truth)
    return {
        "flip_rate":          (fp+fn)           / total if total else float("nan"),
        "weighted_flip_rate": (fp+FN_WEIGHT*fn) / total if total else float("nan"),
        "false_positive_rate": fp / total if total else float("nan"),
        "false_negative_rate": fn / total if total else float("nan"),
    }

def threshold_alarm(series: pd.Series, threshold: float) -> pd.Series:
    return (series > threshold).astype(int)

def trend_positive(series: pd.Series, window_days: int) -> pd.Series:
    """Flag days where every step in the preceding window_days was an increase."""
    labels = pd.Series(0, index=series.index)
    diffs  = series.diff()
    for t in range(window_days, len(series)):
        window = diffs.iloc[t - window_days + 1: t + 1].dropna()
        if len(window) == window_days and (window > 0).all():
            labels.iloc[t] = 1
    return labels

def change_points_by_diff(series: pd.Series, threshold: float) -> pd.Series:
    return (series.diff().abs().fillna(0) > threshold).astype(int)

def cp_f1_with_tolerance(truth_cp: pd.Series, pred_cp: pd.Series, tol_days: int) -> tuple:
    truth_idx = np.where(truth_cp.to_numpy(int) == 1)[0]
    pred_idx  = np.where(pred_cp.to_numpy(int)  == 1)[0]
    if len(truth_idx) == 0 and len(pred_idx) == 0: return (1.0, 0.0)
    if len(truth_idx) == 0: return (0.0, float("nan"))
    if len(pred_idx)  == 0: return (0.0, float("nan"))
    used_truth, tp, delays = set(), 0, []
    for p in pred_idx:
        cands = [t for t in truth_idx if t not in used_truth and abs(t-p) <= tol_days]
        if cands:
            t_best = min(cands, key=lambda t: abs(t-p))
            used_truth.add(t_best); tp += 1; delays.append(p - t_best)
    fp = len(pred_idx) - tp; fn = len(truth_idx) - tp
    prec   = tp / (tp+fp) if (tp+fp) else 0.0
    recall = tp / (tp+fn) if (tp+fn) else 0.0
    f1 = 2*prec*recall/(prec+recall) if (prec+recall) else 0.0
    return (f1, float(np.mean(delays)) if delays else float("nan"))

def overlap_at_k(truth_series: pd.Series, dp_series: pd.Series, k: int) -> float:
    k = min(k, len(truth_series))
    if k == 0: return float("nan")
    return len(set(truth_series.nlargest(k).index) & set(dp_series.nlargest(k).index)) / k

def kendall_tau_on_ranks(truth_series: pd.Series, dp_series: pd.Series) -> float:
    a = truth_series.rank(method="average", ascending=False).to_numpy()
    b = dp_series.rank(method="average", ascending=False).to_numpy()
    n = len(a)
    if n < 2: return float("nan")
    concordant = discordant = 0
    for i in range(n):
        for j in range(i+1, n):
            s1, s2 = np.sign(a[i]-a[j]), np.sign(b[i]-b[j])
            if s1 == 0 or s2 == 0: continue
            if s1 == s2: concordant += 1
            else: discordant += 1
    denom = concordant + discordant
    return (concordant - discordant) / denom if denom else float("nan")

def eval_pointwise_and_trend(truth: pd.Series, dp: pd.Series) -> dict:
    return {
        "MAE":        float(mean_absolute_error(truth, dp)),
        "RMSE":       rmse(truth, dp),
        "CORR_LEVEL": pearson_corr(truth, dp),
        "CORR_DIFF":  pearson_corr(truth.diff().fillna(0), dp.diff().fillna(0)),
    }

def save_json(path: Path, obj: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)

## Disjoint Slice Generation

In [ ]:
def gen_slices_disjoint(min_date, max_date, block_days, max_slices=None):
    slices, start, count = [], min_date, 0
    while start <= max_date:
        end = start + timedelta(days=block_days - 1)
        if end > max_date: break
        slices.append((start, end))
        start = end + timedelta(days=1); count += 1
        if max_slices is not None and count >= max_slices: break
    return slices

slice_jobs = []
for w in WINDOW_DAYS_LIST:
    for (s, e) in gen_slices_disjoint(min_date, max_date, w):
        slice_jobs.append({"window_days": w, "start": s, "end": e})

print(f"Disjoint slice jobs: {len(slice_jobs)}")
for sj in slice_jobs:
    print(f"  block={sj['window_days']}d  {sj['start'].date()} → {sj['end'].date()}")

total_rows = len(slice_jobs)*len(BYTES_CLIP_PCT_LIST)*len(EPS_TOTALS)*3*3*NUM_SEEDS
print(f"\nEstimated result rows: {total_rows:,}")
print(f"  = {len(slice_jobs)} slices × {len(BYTES_CLIP_PCT_LIST)} clip_pcts × "
      f"{len(EPS_TOTALS)} eps × 3 mechs × 3 metrics × {NUM_SEEDS} seeds")

## Grid Search Loop

In [ ]:
rows = []
seed_iter = tqdm(range(NUM_SEEDS), desc="Seeds", unit="seed") if HAS_TQDM else range(NUM_SEEDS)

for seed_id in seed_iter:
    rng = np.random.default_rng(BASE_SEED + seed_id)

    for sj in slice_jobs:
        s, e, wdays = sj["start"], sj["end"], sj["window_days"]

        for bytes_pct in BYTES_CLIP_PCT_LIST:
            daily_truth, B = make_daily_bounded_truth(flows, s, e, bytes_pct)

            sens = {"ALERT_COUNT": 1.0, "TOTAL_FLOWS": 1.0, "TOTAL_BYTES": float(B)}
            bounded_tag = f"s={s.date()}__e={e.date()}__win={wdays}__bytespct={bytes_pct}"

            if seed_id == 0:
                bp = BOUNDED_DIR / f"bounded__{bounded_tag}.parquet"
                daily_truth.to_parquet(bp)
                save_json(bp.with_suffix(".json"), {
                    "strategy": STRATEGY_NAME, "slice_start": str(s.date()),
                    "slice_end": str(e.date()), "window_days": wdays,
                    "bytes_clip_pct": bytes_pct, "B_bytes_cap": B, "sensitivities": sens,
                })

            thresh    = {m: float(daily_truth[m].quantile(THRESH_QUANTILE)) for m in METRICS}
            cp_thresh = {m: CP_DIFF_MULTIPLIER * max(float(daily_truth[m].diff().fillna(0).std()), 1e-6)
                         for m in METRICS}
            topk_eff  = min(TOPK, len(daily_truth))

            for eps in EPS_TOTALS:
                dp_naive  = dp_naive_laplace(daily_truth, eps, sens, rng)
                dp_tree   = dp_binary_tree(daily_truth, eps, sens, rng)
                dp_smooth = dp_smooth_binary(daily_truth, eps, sens, rng)

                dp_pack = {
                    "naive":  (dp_naive,  {m: f"{m}_DP"        for m in METRICS}),
                    "tree":   (dp_tree,   {m: f"{m}_TREE_DP"   for m in METRICS}),
                    "smooth": (dp_smooth, {m: f"{m}_SMOOTH_DP" for m in METRICS}),
                }

                if seed_id == 0:
                    for mech, (dpdf, _) in dp_pack.items():
                        dp_path = DP_DIR / f"dp__{mech}__eps={eps}__{bounded_tag}.parquet"
                        dpdf.to_parquet(dp_path)

                for mech, (dpdf, colmap) in dp_pack.items():
                    for metric in METRICS:
                        truth_s = daily_truth[metric].astype(float)
                        dp_s    = dpdf[colmap[metric]].astype(float)

                        pt = eval_pointwise_and_trend(truth_s, dp_s)

                        alarm_truth = threshold_alarm(truth_s, thresh[metric])
                        alarm_dp    = threshold_alarm(dp_s,    thresh[metric])
                        alarm_stats = decision_stats(alarm_truth, alarm_dp)

                        trend_truth = trend_positive(truth_s, TREND_WINDOW_DAYS)
                        trend_dp    = trend_positive(dp_s,    TREND_WINDOW_DAYS)
                        trend_stats = decision_stats(trend_truth, trend_dp)

                        cp_truth    = change_points_by_diff(truth_s, cp_thresh[metric])
                        cp_dp       = change_points_by_diff(dp_s,    cp_thresh[metric])
                        cp_f1, cp_delay = cp_f1_with_tolerance(cp_truth, cp_dp, CP_TOLERANCE_DAYS)

                        ovk = overlap_at_k(truth_s, dp_s, topk_eff)
                        tau = kendall_tau_on_ranks(truth_s, dp_s)

                        rows.append({
                            "seed":            seed_id,
                            "strategy":        STRATEGY_NAME,
                            "slice_start":     str(s.date()),
                            "slice_end":       str(e.date()),
                            "window_days":     wdays,
                            "bytes_clip_pct":  bytes_pct,
                            "B_bytes_cap":     B,
                            "eps_total":       eps,
                            "mechanism":       mech,
                            "metric":          metric,
                            "MAE":             pt["MAE"],
                            "RMSE":            pt["RMSE"],
                            "CORR_LEVEL":      pt["CORR_LEVEL"],
                            "CORR_DIFF":       pt["CORR_DIFF"],
                            "THRESH":          thresh[metric],
                            "ALARM_flip":          alarm_stats["flip_rate"],
                            "ALARM_weighted_flip": alarm_stats["weighted_flip_rate"],
                            "ALARM_fp_rate":       alarm_stats["false_positive_rate"],
                            "ALARM_fn_rate":       alarm_stats["false_negative_rate"],
                            "TREND_flip":    trend_stats["flip_rate"],
                            "TREND_fp_rate": trend_stats["false_positive_rate"],
                            "TREND_fn_rate": trend_stats["false_negative_rate"],
                            "CP_thresh":    cp_thresh[metric],
                            "CP_f1":        cp_f1,
                            "CP_avg_delay": cp_delay,
                            "TOPK":         topk_eff,
                            "OVERLAP_at_k": ovk,
                            "KENDALL_tau":  tau,
                        })

print(f"Done. Total rows: {len(rows):,}  ({NUM_SEEDS} seeds × {len(rows)//NUM_SEEDS:,} rows/seed)")

In [ ]:
results_df = pd.DataFrame(rows)

# Save as *_hardened.csv to preserve the existing *_multi.csv (4-clip-pct grid)
csv_path = CSV_DIR / "results_disjoint_hardened.csv"
results_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}  shape={results_df.shape}")

seed_counts = results_df.groupby("seed").size()
assert seed_counts.nunique() == 1, "Unequal rows per seed — something went wrong!"
print(f"Rows per seed: {seed_counts.iloc[0]:,}  ✓")

summary = (
    results_df.groupby(["metric","mechanism","eps_total"])["MAE"]
    .agg(["mean","std"]).round(4)
    .rename(columns={"mean":"MAE_mean","std":"MAE_std"})
)
print(f"\nMAE summary (mean ± std, {NUM_SEEDS} seeds):")
display(summary)

# Quick CV check — should be lower than the 7-seed multi run
cv = summary["MAE_std"] / summary["MAE_mean"].replace(0, float("nan"))
print(f"\nMean CV across all rows: {cv.mean():.3f}  (target: < 0.5 for stable estimates)")

In [ ]:
MECH_COLORS = {{"naive":"#e15759","tree":"#4e79a7","smooth":"#59a14f"}}

for metric in METRICS:
    sub = results_df[results_df["metric"]==metric]
    agg = (sub.groupby(["eps_total","mechanism"])["MAE"]
              .agg(["mean","std"]).reset_index())
    fig, ax = plt.subplots(figsize=(7,4))
    for mech, grp in agg.groupby("mechanism"):
        grp = grp.sort_values("eps_total")
        ax.plot(grp["eps_total"], grp["mean"], marker="o",
                label=mech, color=MECH_COLORS[mech], lw=2)
        ax.fill_between(grp["eps_total"],
                        (grp["mean"]-grp["std"]).clip(lower=0),
                        grp["mean"]+grp["std"],
                        alpha=0.20, color=MECH_COLORS[mech])
    ax.set_xscale("log")
    ax.set_xlabel("ε (log scale)")
    ax.set_ylabel(f"MAE (mean ± 1 std, {NUM_SEEDS} seeds)")
    ax.set_title(f"{STRATEGY_NAME} — {metric} MAE vs ε  (n={NUM_SEEDS} seeds, clip=70th pct)")
    ax.legend(); plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"hardened_mae_vs_eps_{metric.lower()}.png", dpi=120)
    plt.close()
    print(f"Saved hardened_mae_vs_eps_{metric.lower()}.png")